# 03 - 前処理段階の診断（どこまで preprocess されたデータか推定）

**ここでは何も揃えない・正規化しない。** 遺伝子ごとの統計量や値の分布を見て、各データが「生カウントか / 正規化済みか / log か / TPM か」を**人間が推定**するための診断のみ。

目安:
- 値が整数 & 最小0 → 生 / filtered UMI count
- 非整数 & 最大値が小さい(≲20) → log 変換後の可能性
- 行和が約 1e6 → CPM/TPM の可能性
- 負値あり → scaling / z-score 後の可能性

In [ ]:
import sys
from pathlib import Path

# config/dataset_manifest.yaml を持つ v2 ルートを探して src を import パスに追加
ROOT = Path.cwd()
while not (ROOT / "config" / "dataset_manifest.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
print("project root:", ROOT)

import manifest_utils as mf
man = mf.load_manifest()
paths = mf.project_paths(ROOT)
mf.ensure_dirs(paths)

import anndata_utils as au
import notebook_report_utils as nb
import matplotlib.pyplot as plt

# curated を見る（まだなら interim に変更可）
adatas = au.load_h5ad_collection(paths['curated'])
if not adatas:
    adatas = au.load_h5ad_collection(paths['interim'])
list(adatas.keys())

## 全データセットの推定一覧

`declared_data_status`（manifest 宣言）と `guess`（値からの推定）を見比べる。

In [ ]:
state = nb.processing_state_table(adatas)
state

## 1データセットを詳しく見る

`key` を変えて再実行。

In [ ]:
key = list(adatas.keys())[0]
a = adatas[key]
nb.infer_processing_state(a)

In [ ]:
# 値・library size・遺伝子数の分布
fig = nb.plot_value_distribution(a, name=key)
plt.show()

In [ ]:
# mean-variance / dropout-mean（過分散や正規化の見当）
fig = nb.plot_gene_stats(a, name=key)
plt.show()

In [ ]:
# 遺伝子ごとの統計量（平均・分散・発現細胞数・dropout・最大）
nb.gene_stats(a).head(20)

## 全データセットの値分布を一気に確認

In [ ]:
for name, adata in adatas.items():
    nb.plot_value_distribution(adata, name=name)
    plt.show()

ここでの推定結果をもとに、merge 時に何を同列に扱えるか（生カウント系だけか等）を判断する。

次: **04_merge_curated_h5ad.ipynb**